# Phase 1: Data Acquisition (Google Earth Engine)
This notebook extracts a time-series dataset including Sentinel-1, Sentinel-2, SMAP, ERA5-Land, Topography, and Soil data using the Earth Engine Python API.

In [1]:
import ee
import geemap

# Authenticate to Earth Engine (uncomment if running for the first time)
# ee.Authenticate()
ee.Initialize()

# Define the user-provided GeoJSON LineString coordinates
coordinates = [
    [83.03748891308771, 24.7757901486428],
    [82.48222230386449, 25.014295467473247],
    [82.30685241896737, 25.76671241133323],
    [82.75252773248934, 26.39013047758114],
    [83.41008316758769, 26.677745920714784],
    [84.73981327571806, 26.455563579381078],
    [85.10514370245755, 25.68114197907728],
    [84.48408272144587, 24.9811658047324],
    [83.59271568205071, 24.649661772243107],
    [83.03742828970536, 24.77577336071262]
]

# Create an ee.Geometry.LineString and use its bounds as our Region of Interest (ROI)
roi = ee.Geometry.LineString(coordinates).bounds()

# Visualization via geemap
Map = geemap.Map()
Map.addLayer(roi, {'color': 'red'}, 'Region of Interest')
Map.centerObject(roi, zoom=8)
display(Map)

start_date = '2021-01-01'
end_date = '2025-12-31'

print('--- Data Collection Summary ---')
print(f'Checking timeframe: {start_date} to {end_date}')

def safe_fetch_count(name, query_func):
    try:
        count = query_func().size().getInfo()
        print(f'{name} images available: {count}')
    except Exception as e:
        msg = str(e).split('\n')[0]
        print(f'{name}: Data not available for the requested timeframe/ROI. (Error: {msg})')

def safe_fetch_stats(name, query_func):
    try:
        stats = query_func().getInfo()
        print(f'{name}: {stats}')
    except Exception as e:
        msg = str(e).split('\n')[0]
        print(f'{name}: Data not available. (Error: {msg})')

# 1. Sentinel-1 Backscatter
safe_fetch_count(
    'Sentinel-1',
    lambda: ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')).filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)

# 2. Sentinel-2 Multispectral
safe_fetch_count(
    'Sentinel-2 (Cloud < 20%)',
    lambda: ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

# 3. SMAP Soil Moisture
# Using NASA/SMAP/SPL4SMGP/007 (L4 Global, 9km, 3-hourly) — NRT asset was removed by NASA
safe_fetch_count(
    'SMAP soil moisture (L4 Global)',
    lambda: ee.ImageCollection('NASA/SMAP/SPL4SMGP/007').filterBounds(roi).filterDate(start_date, end_date)
)

# 4. ERA5-Land
safe_fetch_count(
    'ERA5-Land hourly',
    lambda: ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterBounds(roi).filterDate(start_date, end_date)
)

# 5. Topography (SRTM)
safe_fetch_stats(
    'Elevation min/max (m) in ROI',
    lambda: ee.Image('USGS/SRTMGL1_003').select('elevation').reduceRegion(reducer=ee.Reducer.minMax(), geometry=roi, scale=100)
)

# 6. Soil Properties (OpenLandMap)
safe_fetch_stats(
    'Mean Clay Fraction at surface in ROI',
    lambda: ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02').select('b0').reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=250)
)


Map(center=[25.663052155712947, 83.70599806071256], controls=(WidgetControl(options=['position', 'transparent_…

--- Data Collection Summary ---
Checking timeframe: 2021-01-01 to 2025-12-31
Sentinel-1 images available: 2063
Sentinel-2 (Cloud < 20%) images available: 4390
SMAP soil moisture (L4 Global) images available: 13110
ERA5-Land hourly images available: 43800
Elevation min/max (m) in ROI: {'elevation_max': 582, 'elevation_min': 31}
Mean Clay Fraction at surface in ROI: {'b0': 26.871170511162177}


In [3]:
import pandas as pd
import numpy as np
import datetime
import ee
import time

def get_yearly_data(year, end_date_str):
    y_start = ee.Date(f'{year}-01-01')
    if year == int(end_date_str.split('-')[0]):
        y_end_exclusive = ee.Date(end_date_str).advance(1, 'day')
    else:
        y_end_exclusive = ee.Date(f'{year+1}-01-01')
    
    numberOfDays = y_end_exclusive.difference(y_start, 'days')
    days = ee.List.sequence(0, numberOfDays.subtract(1))
    
    def extract_daily_features(day_offset):
        date = y_start.advance(ee.Number(day_offset), 'day')
        next_date = date.advance(1, 'day')
        
        # --- Dynamic Forcings ---
        era5 = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
            .filterBounds(roi) \
            .filterDate(date, next_date)
        precip = era5.select('total_precipitation').sum().rename('total_precipitation')
        temp = era5.select('temperature_2m').mean().rename('temperature_2m')
        pe = era5.select('potential_evaporation').sum().rename('potential_evaporation')
        
        # SMAP
        smap = ee.ImageCollection('NASA/SMAP/SPL4SMGP/007') \
            .filterBounds(roi) \
            .filterDate(date, next_date) \
            .select('sm_surface') \
            .mean().rename('sm_surface')
        
        # Sentinel-1
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterBounds(roi) \
            .filterDate(date, next_date) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
            .select(['VV', 'VH']) \
            .mean()
        
        # Sentinel-2 (NDVI)
        s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
            .filterBounds(roi) \
            .filterDate(date, next_date) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        def compute_ndvi(img):
            return img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndvi = s2.map(compute_ndvi).mean()
        s2_bands = s2.select(['B2', 'B3', 'B4', 'B8']).mean()
        
        daily_img = ee.Image().addBands([precip, temp, pe, smap, s1, ndvi, s2_bands])
        stats = daily_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=1000,
            maxPixels=1e9
        )
        return ee.Feature(None, stats).set('date', date.format('YYYY-MM-dd'))
    
    daily_fc = ee.FeatureCollection(days.map(extract_daily_features))
    df = geemap.ee_to_df(daily_fc)
    return df

all_dfs = []
start_yr = int(start_date.split('-')[0])
end_yr = int(end_date.split('-')[0])

for year in range(start_yr, end_yr + 1):
    print(f"Extracting daily time-series for {year} (this may take a minute)...")
    df_year = get_yearly_data(year, end_date)
    all_dfs.append(df_year)
    time.sleep(2) # brief pause between big requests

df_dynamic = pd.concat(all_dfs, ignore_index=True)
df_dynamic['date'] = pd.to_datetime(df_dynamic['date'])
df_dynamic = df_dynamic.sort_values('date').reset_index(drop=True)

print("Extraction complete. First 5 rows of raw dynamic data:")
display(df_dynamic.head())

# --- 2. Extract Static Context ---
print("\nExtracting static context...")
elev = ee.Image('USGS/SRTMGL1_003').select('elevation')
slope = ee.Terrain.slope(elev).rename('slope')
clay = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('clay')
sand = ee.Image('OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('sand')

static_img = ee.Image().addBands([elev, slope, clay, sand])
static_stats = static_img.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=1000,
    maxPixels=1e9
).getInfo()

print("Static variables for ROI:")
print(static_stats)

# --- 3. Interpolation for Gap-filling ---
print("\nInterpolating gaps...")
cols_to_interpolate = ['VV', 'VH', 'NDVI', 'B2', 'B3', 'B4', 'B8']
existing_cols = [c for c in cols_to_interpolate if c in df_dynamic.columns]
if existing_cols:
    df_dynamic[existing_cols] = df_dynamic[existing_cols].interpolate(method='linear', limit_direction='both')

# --- 4. Append Static Context ---
for key, val in static_stats.items():
    df_dynamic[key] = val

print("\nFinal Aligned and Interpolated Dataset:")
display(df_dynamic.head())

# --- 5. Export ---
output_file = f'roi_aligned_timeseries_{start_yr}_{end_yr}.csv'
df_dynamic.to_csv(output_file, index=False)
print(f"\nDataset successfully exported to: {output_file}")


Extracting daily time-series for 2021 (this may take a minute)...
Extracting daily time-series for 2022 (this may take a minute)...
Extracting daily time-series for 2023 (this may take a minute)...
Extracting daily time-series for 2024 (this may take a minute)...
Extracting daily time-series for 2025 (this may take a minute)...
Extraction complete. First 5 rows of raw dynamic data:


,date,potential_evaporation,sm_surface,temperature_2m,total_precipitation,B2,B3,B4,B8,NDVI,VH,VV
0,2021-01-01,-0.064895,0.159273,287.775122,8.557963e-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-02,-0.059693,0.158111,289.226754,2.312732e-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-03,-0.059155,0.158204,290.845856,7.735890e-06,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761
3,2021-01-04,-0.060549,0.172161,292.214553,9.411331e-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-05,-0.042654,0.175980,292.534701,3.264632e-04,1068.659750,1205.111047,1197.879132,2102.806262,0.289162,-17.437728,-10.704155



Extracting static context...
Static variables for ROI:
{'clay': 27.11630870032713, 'constant': None, 'elevation': 98.5503379392352, 'sand': 37.425354777988694, 'slope': 0.19741518287534388}

Interpolating gaps...

Final Aligned and Interpolated Dataset:


,date,potential_evaporation,sm_surface,temperature_2m,total_precipitation,B2,B3,B4,B8,NDVI,VH,VV,clay,constant,elevation,sand,slope
0,2021-01-01,-0.064895,0.159273,287.775122,8.557963e-07,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761,27.116309,None,98.550338,37.425355,0.197415
1,2021-01-02,-0.059693,0.158111,289.226754,2.312732e-07,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761,27.116309,None,98.550338,37.425355,0.197415
2,2021-01-03,-0.059155,0.158204,290.845856,7.735890e-06,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761,27.116309,None,98.550338,37.425355,0.197415
3,2021-01-04,-0.060549,0.172161,292.214553,9.411331e-06,1106.706116,1268.856384,1182.136765,2244.874586,0.317548,-17.412232,-10.193458,27.116309,None,98.550338,37.425355,0.197415
4,2021-01-05,-0.042654,0.175980,292.534701,3.264632e-04,1068.659750,1205.111047,1197.879132,2102.806262,0.289162,-17.437728,-10.704155,27.116309,None,98.550338,37.425355,0.197415



Dataset successfully exported to: roi_aligned_timeseries_2021_2025.csv
